# 02: Reversible transformer at the same batch size
**Step 1: variant pilots.** Each variant (plus a baseline reference) is trained for 5M tokens at the baseline batch size:
- `euler`: two-stream additive coupling (RevNet/Reformer, a symplectic-Euler step), h=1
- `midpoint`: leapfrog/explicit-midpoint step `(a, b) → (b, a + 2h·F(b))`, h=0.5 and h=0.25

**Step 2:** full 50M-token run of the variant with the best pilot validation loss.

## Setup
Runtime → Change runtime type → **T4 GPU**. Results are written to Google Drive, so a disconnect does not lose finished runs.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/reversible-llm'
import os
os.makedirs(f'{DRIVE}/data', exist_ok=True)
os.makedirs(f'{DRIVE}/results', exist_ok=True)

In [ ]:
import os, sys
if not os.path.exists('/content/llm_finetune_assignments'):
    !git clone -q https://github.com/AditeyaAItronics/llm_finetune_assignments.git /content/llm_finetune_assignments
%cd /content/llm_finetune_assignments/reversible-llm-13
!git pull -q
!pip install -q tiktoken datasets
sys.path.insert(0, '/content/llm_finetune_assignments/reversible-llm-13/src')

In [ ]:
from revllm.data import prepare_tinystories
prepare_tinystories(f'{DRIVE}/data')  # tokenizes once (~55M tokens), cached on Drive
!mkdir -p /content/data && cp -n {DRIVE}/data/*.bin /content/data/

In [ ]:
import pytest
# includes the GPU memory test that is skipped on CPU
assert pytest.main(['-q', 'tests']) == 0, 'tests failed: fix before training'

In [ ]:
import json
from revllm.config import ModelConfig, TrainConfig
from revllm.train import train

B = json.load(open(f'{DRIVE}/results/01_baseline.json'))['batch_size']
PILOTS = [('baseline', None), ('euler', 1.0), ('midpoint', 0.5), ('midpoint', 0.25)]
pilots = {}
for arch, h in PILOTS:
    mc = ModelConfig(arch=arch, h=h)
    name = f'02_pilot_{arch}_h{mc.h}'
    pilots[name] = train(mc, TrainConfig(run_name=name, total_tokens=5_000_000, batch_size=B,
                                         eval_every_tokens=1_000_000,
                                         data_dir='/content/data', out_dir=f'{DRIVE}/results'))
KEYS = ['diverged', 'final_val_loss', 'tokens_per_sec', 'peak_mem_allocated_gb', 'recon_error']
{n: {k: r[k] for k in KEYS} for n, r in pilots.items()}

In [ ]:
ok = [r for r in pilots.values() if r['arch'] != 'baseline' and not r['diverged']]
if not ok:
    raise RuntimeError('all reversible pilots diverged - rerun 02 with lower lr or h')
winner = min(ok, key=lambda r: r['final_val_loss'])
json.dump({'arch': winner['arch'], 'h': winner['h']}, open(f'{DRIVE}/winner.json', 'w'))
print('Winner:', winner['arch'], 'h =', winner['h'])

In [ ]:
mc = ModelConfig(arch=winner['arch'], h=winner['h'])
res = train(mc, TrainConfig(run_name=f"02_reversible_{winner['arch']}_same_batch", batch_size=B,
                            data_dir='/content/data', out_dir=f'{DRIVE}/results'))
{k: res[k] for k in ['batch_size', 'tokens_seen', 'final_train_loss', 'final_val_loss',
                     'tokens_per_sec', 'peak_mem_allocated_gb', 'wall_time_s', 'recon_error']}